In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')
products = pd.read_csv(BASE_PATH + 'products.csv')

# keep only Delivered orders, then attach order date and branch to each line
delivered = so_h[so_h['order_status'] == 'Delivered']
sales = so_l[so_l['so_id'].isin(delivered['so_id'])].merge(
    delivered[['so_id', 'order_date', 'branch_id']], on='so_id', how='left')
sales['order_date'] = pd.to_datetime(sales['order_date'])

print('so_h shape:', so_h.shape, '| expected (20000, 11)')
print('so_l shape:', so_l.shape, '| expected (130402, 9)')
print('products shape:', products.shape, '| expected (30, 23)')
print('Delivered orders:', len(delivered), '| expected 18033')
print('sales (Delivered lines):', len(sales), '| expected 117656')
print('units in sales:', sales['quantity'].sum(), '| expected 1234939')
print('first order date:', sales['order_date'].min().date(), '| last:', sales['order_date'].max().date())

Mounted at /content/drive
so_h shape: (20000, 11) | expected (20000, 11)
so_l shape: (130402, 9) | expected (130402, 9)
products shape: (30, 23) | expected (30, 23)
Delivered orders: 18033 | expected 18033
sales (Delivered lines): 117656 | expected 117656
units in sales: 1234939 | expected 1234939
first order date: 2019-01-01 | last: 2024-12-31


In [2]:
# units sold per product and branch for each month, Delivered lines only
sales['month'] = sales['order_date'].dt.to_period('M')
monthly = sales.groupby(['product_id', 'branch_id', 'month'])['quantity'].sum().unstack('month', fill_value=0)

print('monthly table shape (pairs x months):', monthly.shape, '| expected (180, 72)')
print('grand total units:', monthly.values.sum(), '| expected 1234939')
print('first month:', monthly.columns[0], '| last month:', monthly.columns[-1])
print('months with zero sales (cells):', (monthly == 0).sum().sum(), 'of', monthly.shape[0] * monthly.shape[1])
print('smallest month for any pair:', monthly.values.min(), '| largest:', monthly.values.max())

monthly table shape (pairs x months): (180, 72) | expected (180, 72)
grand total units: 1234939 | expected 1234939
first month: 2019-01 | last month: 2024-12
months with zero sales (cells): 5 of 12960
smallest month for any pair: 0 | largest: 347


In [3]:
# find the pair-months with no sales
zero_cells = monthly.stack()
zero_cells = zero_cells[zero_cells == 0].reset_index()
zero_cells.columns = ['product_id', 'branch_id', 'month', 'units']
print(zero_cells.to_string(index=False))
print('number of zero cells:', len(zero_cells), '| expected 5')

product_id branch_id   month  units
      P004    HYD001 2021-01      0
      P004    HYD001 2024-06      0
      P008    KOL001 2023-02      0
      P018    HYD001 2020-02      0
      P021    HYD001 2023-08      0
number of zero cells: 5 | expected 5


In [4]:
# units per product for each year, then compare first three years with last three years
yearly = monthly.T.groupby(monthly.columns.year).sum().T
by_product = yearly.groupby(level='product_id').sum()

print('units per year, all products:')
print(yearly.sum().to_string())

first3 = by_product[[2019, 2020, 2021]].sum(axis=1)
last3 = by_product[[2022, 2023, 2024]].sum(axis=1)
overall_change = (last3.sum() / first3.sum() - 1) * 100
print('overall change, 2019-21 vs 2022-24 (percent):', round(overall_change, 2), '| KPI 5 reported about -3.4')

change = (last3 / first3 - 1) * 100
print('products that rose:', (change > 0).sum(), '| products that fell:', (change < 0).sum(), '| of', len(change))
print('biggest rise:', change.idxmax(), round(change.max(), 2), '| biggest fall:', change.idxmin(), round(change.min(), 2))

units per year, all products:
month
2019    207925
2020    212007
2021    208269
2022    202242
2023    204638
2024    199858
overall change, 2019-21 vs 2022-24 (percent): -3.42 | KPI 5 reported about -3.4
products that rose: 6 | products that fell: 24 | of 30
biggest rise: P006 4.27 | biggest fall: P028 -9.52


In [5]:
# units for each calendar month in each year
by_month = monthly.sum().to_frame('units')
by_month['year'] = by_month.index.year
by_month['calendar_month'] = by_month.index.month
table = by_month.pivot(index='calendar_month', columns='year', values='units')
print('units by calendar month and year:')
print(table.to_string())

# share of each year's units that falls in each calendar month
share = table / table.sum() * 100
print('share of the year in each calendar month (percent):')
print(share.round(2).to_string())
print('average share by calendar month (percent):')
print(share.mean(axis=1).round(2).to_string())

print('busiest calendar month in each year:')
print(table.idxmax().to_string())
print('quietest calendar month in each year:')
print(table.idxmin().to_string())

units by calendar month and year:
year             2019   2020   2021   2022   2023   2024
calendar_month                                          
1               17188  16441  18203  18717  15630  17745
2               14117  15393  17227  15212  15194  16222
3               18013  16317  16612  16703  18548  17799
4               17860  16060  16107  16233  17500  14911
5               18496  19488  16604  18829  16681  15200
6               16734  17130  15816  15361  16004  16988
7               19642  22304  17061  16843  17955  16665
8               19033  20237  18048  17551  17830  17544
9               16830  16503  16378  16048  19129  17650
10              16823  16247  19685  17541  16631  17197
11              17304  18034  17716  16786  16506  16501
12              15885  17853  18812  16418  17030  15436
share of the year in each calendar month (percent):
year            2019   2020  2021  2022  2023  2024
calendar_month                                     
1           

In [6]:
# split the monthly table into training months and the hidden year
train = monthly.loc[:, monthly.columns <= pd.Period('2023-12', 'M')]
test = monthly.loc[:, monthly.columns >= pd.Period('2024-01', 'M')]

print('training months:', train.shape[1], '| test months:', test.shape[1], '| expected 60 and 12')
print('training units + test units:', train.values.sum() + test.values.sum(), '| expected 1234939')
print('zero cells in training:', (train == 0).sum().sum(), '| expected 4')
print('zero cells in test:', (test == 0).sum().sum(), '| expected 1')
print('training units:', train.values.sum(), '| test units:', test.values.sum(), '| test should equal 199858')

training months: 60 | test months: 12 | expected 60 and 12
training units + test units: 1234939 | expected 1234939
zero cells in training: 4 | expected 4
zero cells in test: 1 | expected 1
training units: 1035081 | test units: 199858 | test should equal 199858


In [7]:
# average of the last 12 training months, and average of all 60 training months, for each pair
recent_avg = train.iloc[:, -12:].mean(axis=1)
all_avg = train.mean(axis=1)

forecast_recent = pd.DataFrame({m: recent_avg for m in test.columns})
forecast_long = pd.DataFrame({m: all_avg for m in test.columns})
forecast_last_year = pd.DataFrame(train.iloc[:, -12:].values, index=train.index, columns=test.columns)

actual_total = test.values.sum()
expected_totals = {'recent level': 204638, 'same as last year': 204638, 'long-run level': 207016}

for name, f in [('recent level', forecast_recent), ('same as last year', forecast_last_year), ('long-run level', forecast_long)]:
    gap = (test - f).abs().values.sum()
    print(name)
    print('  forecast total units:', round(f.values.sum()), '| expected about', expected_totals[name])
    print('  forecast total minus actual total:', round(f.values.sum() - actual_total))
    print('  total absolute gap:', round(gap), '| as percent of actual units:', round(gap / actual_total * 100, 2))


recent level
  forecast total units: 204638 | expected about 204638
  forecast total minus actual total: 4780
  total absolute gap: 67159 | as percent of actual units: 33.6
same as last year
  forecast total units: 204638 | expected about 204638
  forecast total minus actual total: 4780
  total absolute gap: 88778 | as percent of actual units: 44.42
long-run level
  forecast total units: 207016 | expected about 207016
  forecast total minus actual total: 7158
  total absolute gap: 64359 | as percent of actual units: 32.2


In [8]:
# compare forecast and actual when months are added up: by month, by quarter, and for the whole year
quarters = [list(test.columns[i:i + 3]) for i in range(0, 12, 3)]
actual_total = test.values.sum()
earlier_monthly = {'recent level': 33.6, 'same as last year': 44.42, 'long-run level': 32.2}

for name, f in [('recent level', forecast_recent), ('same as last year', forecast_last_year), ('long-run level', forecast_long)]:
    month_gap = (test - f).abs().values.sum()
    quarter_gap = 0
    for q in quarters:
        quarter_gap += (test[q].sum(axis=1) - f[q].sum(axis=1)).abs().sum()
    year_gap = (test.sum(axis=1) - f.sum(axis=1)).abs().sum()
    print(name)
    print('  gap by month, percent of actual:', round(month_gap / actual_total * 100, 2), '| earlier result', earlier_monthly[name])
    print('  gap by quarter, percent of actual:', round(quarter_gap / actual_total * 100, 2))
    print('  gap for the full year, percent of actual:', round(year_gap / actual_total * 100, 2))

recent level
  gap by month, percent of actual: 33.6 | earlier result 33.6
  gap by quarter, percent of actual: 21.14
  gap for the full year, percent of actual: 14.41
same as last year
  gap by month, percent of actual: 44.42 | earlier result 44.42
  gap by quarter, percent of actual: 26.96
  gap for the full year, percent of actual: 14.41
long-run level
  gap by month, percent of actual: 32.2 | earlier result 32.2
  gap by quarter, percent of actual: 18.7
  gap for the full year, percent of actual: 11.06


In [9]:
import numpy as np

# fit one straight line per pair on the 60 training months, then project the 12 test months
x_train = np.arange(60)
x_test = np.arange(60, 72)
slope, intercept = np.polyfit(x_train, train.values.T, 1)
trend_values = np.outer(slope, x_test) + intercept[:, None]
trend_values = np.clip(trend_values, 0, None)
forecast_trend = pd.DataFrame(trend_values, index=train.index, columns=test.columns)

quarters = [list(test.columns[i:i + 3]) for i in range(0, 12, 3)]
actual_total = test.values.sum()

print('pairs with a falling line:', (slope < 0).sum(), 'of', len(slope))
for name, f in [('straight-line trend', forecast_trend), ('long-run level', forecast_long)]:
    month_gap = (test - f).abs().values.sum()
    quarter_gap = 0
    for q in quarters:
        quarter_gap += (test[q].sum(axis=1) - f[q].sum(axis=1)).abs().sum()
    year_gap = (test.sum(axis=1) - f.sum(axis=1)).abs().sum()
    print(name)
    print('  forecast total units:', round(f.values.sum()), '| minus actual total:', round(f.values.sum() - actual_total))
    print('  gap by month, percent of actual:', round(month_gap / actual_total * 100, 2))
    print('  gap by quarter, percent of actual:', round(quarter_gap / actual_total * 100, 2))
    print('  gap for the full year, percent of actual:', round(year_gap / actual_total * 100, 2))
print('long-run level earlier results: 32.2 by month, 18.7 by quarter, 11.06 for the full year, total 207016')

pairs with a falling line: 103 of 180
straight-line trend
  forecast total units: 203656 | minus actual total: 3798
  gap by month, percent of actual: 32.99
  gap by quarter, percent of actual: 20.07
  gap for the full year, percent of actual: 12.98
long-run level
  forecast total units: 207016 | minus actual total: 7158
  gap by month, percent of actual: 32.2
  gap by quarter, percent of actual: 18.7
  gap for the full year, percent of actual: 11.06
long-run level earlier results: 32.2 by month, 18.7 by quarter, 11.06 for the full year, total 207016


In [10]:
# accuracy for each product: forecast for 2024 against actual 2024, branches added together
actual_by_product = test.sum(axis=1).groupby(level='product_id').sum()
forecast_by_product = forecast_long.sum(axis=1).groupby(level='product_id').sum()
signed_percent = (forecast_by_product - actual_by_product) / actual_by_product * 100
gap_percent = signed_percent.abs()

result = pd.DataFrame({
    'actual': actual_by_product,
    'forecast': forecast_by_product.round(),
    'forecast minus actual percent': signed_percent.round(2),
    'gap percent': gap_percent.round(2),
})
print(result.sort_values('gap percent').to_string())

print('sum of actual:', result['actual'].sum(), '| expected 199858')
print('sum of forecast:', round(forecast_by_product.sum()), '| expected 207016')
print('products with a gap under 5 percent:', (gap_percent < 5).sum())
print('products with a gap under 10 percent:', (gap_percent < 10).sum())
print('products with a gap of 20 percent or more:', (gap_percent >= 20).sum())
print('products forecast too high:', (signed_percent > 0).sum(), '| too low:', (signed_percent < 0).sum())
print('total gap across products, percent of actual:', round((forecast_by_product - actual_by_product).abs().sum() / actual_by_product.sum() * 100, 2))

            actual  forecast  forecast minus actual percent  gap percent
product_id                                                              
P017          6841    6816.0                          -0.37         0.37
P016          6985    6957.0                          -0.40         0.40
P014          6788    6757.0                          -0.45         0.45
P013          6702    6645.0                          -0.85         0.85
P024          6638    6570.0                          -1.03         1.03
P011          6925    7034.0                           1.57         1.57
P008          6739    6864.0                           1.85         1.85
P027          6931    6788.0                          -2.07         2.07
P021          6844    7004.0                           2.34         2.34
P004          6680    6842.0                           2.43         2.43
P006          7149    6939.0                          -2.94         2.94
P030          7141    6891.0                       

In [11]:
# forecast for 2025: average month over all 72 months for each pair, times 12
pair_average = monthly.mean(axis=1)
forecast_2025 = pair_average * 12
by_product_2025 = forecast_2025.groupby(level='product_id').sum()
by_branch_2025 = forecast_2025.groupby(level='branch_id').sum()

print('pairs forecast:', len(forecast_2025), '| expected 180')
print('forecast total units for 2025:', round(forecast_2025.sum()), '| expected about 205823')
print('2025 forecast minus 2024 actual (199858):', round(forecast_2025.sum() - 199858))
print('forecast by branch:')
print(by_branch_2025.round().to_string())
print('forecast by product:')
print(by_product_2025.round().to_string())
print('smallest product:', by_product_2025.idxmin(), round(by_product_2025.min()), '| largest product:', by_product_2025.idxmax(), round(by_product_2025.max()))

pairs forecast: 180 | expected 180
forecast total units for 2025: 205823 | expected about 205823
2025 forecast minus 2024 actual (199858): 5965
forecast by branch:
branch_id
AHM001    35704.0
CHN001    42320.0
DEL001    30886.0
HYD001    28162.0
KOL001    38478.0
PUN001    30272.0
forecast by product:
product_id
P001    6884.0
P002    6820.0
P003    6952.0
P004    6815.0
P005    6747.0
P006    6974.0
P007    7026.0
P008    6843.0
P009    6758.0
P010    7049.0
P011    7016.0
P012    6925.0
P013    6654.0
P014    6762.0
P015    6802.0
P016    6962.0
P017    6820.0
P018    6792.0
P019    6924.0
P020    6819.0
P021    6977.0
P022    6883.0
P023    6713.0
P024    6581.0
P025    6848.0
P026    6734.0
P027    6812.0
P028    6882.0
P029    7117.0
P030    6932.0
smallest product: P024 6581 | largest product: P029 7117


In [12]:
# year miss for each pair: how many pairs are too high, too low, and how much is the overall level
pair_actual = test.sum(axis=1)
pair_forecast = forecast_long.sum(axis=1)
diff = pair_forecast - pair_actual

print('pairs forecast too high:', (diff > 0).sum(), '| too low:', (diff < 0).sum(), '| of', len(diff))
print('units over-forecast in the pairs that are too high:', round(diff[diff > 0].sum()))
print('units under-forecast in the pairs that are too low:', round(-diff[diff < 0].sum()))
print('net difference:', round(diff.sum()), '| expected 7158')
print('year gap per pair, percent of actual:', round(diff.abs().sum() / pair_actual.sum() * 100, 2), '| earlier result 11.06')

# remove the overall over-forecast from every pair, then measure the gap again
rescaled = pair_forecast * (pair_actual.sum() / pair_forecast.sum())
print('year gap per pair after removing the overall level, percent of actual:', round((rescaled - pair_actual).abs().sum() / pair_actual.sum() * 100, 2))

pairs forecast too high: 104 | too low: 76 | of 180
units over-forecast in the pairs that are too high: 14630
units under-forecast in the pairs that are too low: 7471
net difference: 7158 | expected 7158
year gap per pair, percent of actual: 11.06 | earlier result 11.06
year gap per pair after removing the overall level, percent of actual: 10.65


In [13]:
from scipy.stats import spearmanr

# how much each product fell, using training years only
yearly_train = train.T.groupby(train.columns.year).sum().T.groupby(level='product_id').sum()
early = yearly_train[[2019, 2020, 2021]].mean(axis=1)
late = yearly_train[[2022, 2023]].mean(axis=1)
change_percent = (late / early - 1) * 100

rho, p_value = spearmanr(change_percent, signed_percent.loc[change_percent.index])
print('products that fell in the training years:', (change_percent < 0).sum(), 'of', len(change_percent))
print('correlation between change and forecast error (Spearman):', round(rho, 3), '| p value:', round(p_value, 4))

four = pd.DataFrame({'change percent': change_percent.round(2), 'forecast minus actual percent': signed_percent.round(2)})
print(four.loc[['P019', 'P005', 'P012', 'P010']].to_string())

products that fell in the training years: 23 of 30
correlation between change and forecast error (Spearman): -0.303 | p value: 0.1033
            change percent  forecast minus actual percent
product_id                                               
P019                 -2.59                          15.67
P005                 -5.93                          11.99
P012                 -4.12                          11.63
P010                 -3.37                          11.08


In [14]:
# blank, zero and negative values in the columns the forecast uses
for col in ['so_id', 'product_id', 'branch_id', 'order_date', 'quantity']:
    print(col, '| blanks:', sales[col].isnull().sum())

print('quantity zero values:', (sales['quantity'] == 0).sum(), '| negative values:', (sales['quantity'] < 0).sum())
print('quantity smallest:', sales['quantity'].min(), '| largest:', sales['quantity'].max())

print('monthly table blanks:', monthly.isnull().sum().sum(), '| negative cells:', (monthly < 0).sum().sum(), '| zero cells:', (monthly == 0).sum().sum(), '| expected zero cells 5')

for name, f in [('recent level', forecast_recent), ('same as last year', forecast_last_year), ('long-run level', forecast_long), ('straight-line trend', forecast_trend)]:
    print(name, '| blanks:', f.isnull().sum().sum(), '| negative cells:', (f < 0).sum().sum())

print('2025 forecast blanks:', forecast_2025.isnull().sum(), '| negative values:', (forecast_2025 < 0).sum())

so_id | blanks: 0
product_id | blanks: 0
branch_id | blanks: 0
order_date | blanks: 0
quantity | blanks: 0
quantity zero values: 0 | negative values: 0
quantity smallest: 1 | largest: 20
monthly table blanks: 0 | negative cells: 0 | zero cells: 5 | expected zero cells 5
recent level | blanks: 0 | negative cells: 0
same as last year | blanks: 0 | negative cells: 0
long-run level | blanks: 0 | negative cells: 0
straight-line trend | blanks: 0 | negative cells: 0
2025 forecast blanks: 0 | negative values: 0


In [15]:
# quiet months: smallest totals and how many pairs have a quiet month
print('smallest 2024 total for any pair:', test.sum(axis=1).min())
print('smallest 72-month total for any pair:', monthly.sum(axis=1).min())
print('pairs with at least one quiet month:', (monthly == 0).any(axis=1).sum(), '| expected 4')

# orders that were not Delivered and are left out
other_orders = so_h[so_h['order_status'] != 'Delivered']
other_lines = so_l[so_l['so_id'].isin(other_orders['so_id'])]
print('orders not Delivered:', len(other_orders), '| expected 1967')
print('statuses of those orders:', other_orders['order_status'].value_counts().to_dict())
print('lines on those orders:', len(other_lines), '| units on those lines:', other_lines['quantity'].sum())
print('lines in sales that belong to those orders:', sales['so_id'].isin(other_orders['so_id']).sum(), '| expected 0')
print('sales lines plus lines on other orders:', len(sales) + len(other_lines), '| all lines in so_l:', len(so_l))

# the same product listed more than once inside one order
repeat = sales.groupby(['so_id', 'product_id']).size()
print('order and product combinations:', len(repeat), '| with more than one line:', (repeat > 1).sum())
print('lines that belong to a repeated combination:', repeat[repeat > 1].sum(), 'of', len(sales))

smallest 2024 total for any pair: 733
smallest 72-month total for any pair: 5081
pairs with at least one quiet month: 4 | expected 4
orders not Delivered: 1967 | expected 1967
statuses of those orders: {'Cancelled': 1967}
lines on those orders: 12746 | units on those lines: 133595
lines in sales that belong to those orders: 0 | expected 0
sales lines plus lines on other orders: 130402 | all lines in so_l: 130402
order and product combinations: 104418 | with more than one line: 12184
lines that belong to a repeated combination: 25422 of 117656


In [16]:
from scipy.stats import binomtest

# each order once in the header, each line once in the lines table
print('orders in header:', len(so_h), '| distinct order ids:', so_h['so_id'].nunique(), '| expected 20000 and 20000')
print('columns of the lines table:', list(so_l.columns))
id_like = [c for c in so_l.columns if so_l[c].nunique() == len(so_l)]
print('columns that are different on every line:', id_like)
for c in id_like:
    print(c, '| repeated values:', so_l[c].duplicated().sum(), '| expected 0')

# lines that are identical to another line apart from the line id, inside the Delivered lines
content_cols = [c for c in sales.columns if c not in id_like]
print('lines identical to another line apart from the line id:', sales.duplicated(subset=content_cols, keep=False).sum(), 'of', len(sales))

# same product twice in one order: how often the two quantities are equal, against chance
combos = sales.groupby(['so_id', 'product_id'])['quantity'].agg(['size', 'nunique'])
two_lines = combos[combos['size'] == 2]
same_quantity = int((two_lines['nunique'] == 1).sum())
shares = sales['quantity'].value_counts(normalize=True)
chance = float((shares ** 2).sum())
result = binomtest(same_quantity, len(two_lines), chance)
print('combinations with exactly two lines:', len(two_lines))
print('two lines with the same quantity:', same_quantity, '=', round(same_quantity / len(two_lines) * 100, 2), 'percent')
print('chance of the same quantity:', round(chance * 100, 2), 'percent | p value:', round(result.pvalue, 4))

orders in header: 20000 | distinct order ids: 20000 | expected 20000 and 20000
columns of the lines table: ['so_id', 'line_number', 'product_id', 'quantity', 'unit_price', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']
columns that are different on every line: []
lines identical to another line apart from the line id: 0 of 117656
combinations with exactly two lines: 11202
two lines with the same quantity: 547 = 4.88 percent
chance of the same quantity: 5.0 percent | p value: 0.5879


In [17]:
# each line is identified by order id and line number: is any pair used twice
print('lines with a repeated order id and line number:', so_l.duplicated(subset=['so_id', 'line_number']).sum(), '| expected 0')

# Delivered lines that match another line in every column except the line number
content_cols = [c for c in sales.columns if c != 'line_number']
print('lines identical to another line apart from the line number:', sales.duplicated(subset=content_cols, keep=False).sum(), 'of', len(sales))
print('extra copies if one of each group is kept:', sales.duplicated(subset=content_cols, keep='first').sum(), '| at least 547 expected from the equal-quantity pairs')

lines with a repeated order id and line number: 0 | expected 0
lines identical to another line apart from the line number: 1374 of 117656
extra copies if one of each group is kept: 688 | at least 547 expected from the equal-quantity pairs


In [18]:
# totals that must agree everywhere the same sales appear
print('Delivered lines:', len(sales), '| expected 117656')
print('units on Delivered lines:', sales['quantity'].sum(), '| expected 1234939')
print('Delivered orders with at least one line:', sales['so_id'].nunique(), '| Delivered orders in the header:', len(delivered), '| expected 18033 and 18033')

print('units in the monthly table:', monthly.values.sum())
print('units in training plus test:', train.values.sum() + test.values.sum())
print('units by year added up:', yearly.sum().sum())

# units by product, branch and pair: once from the sales lines, once from the monthly table
by_product_sales = sales.groupby('product_id')['quantity'].sum()
by_product_table = monthly.sum(axis=1).groupby(level='product_id').sum()
print('products:', len(by_product_sales), '| largest difference by product:', (by_product_sales - by_product_table).abs().max())

by_branch_sales = sales.groupby('branch_id')['quantity'].sum()
by_branch_table = monthly.sum(axis=1).groupby(level='branch_id').sum()
print('branches:', len(by_branch_sales), '| largest difference by branch:', (by_branch_sales - by_branch_table).abs().max())

by_pair_sales = sales.groupby(['product_id', 'branch_id'])['quantity'].sum()
by_pair_table = monthly.sum(axis=1)
print('pairs:', len(by_pair_sales), '| largest difference by pair:', (by_pair_sales - by_pair_table).abs().max())

# forecast totals
print('2025 forecast added up by product:', round(by_product_2025.sum(), 2), '| by branch:', round(by_branch_2025.sum(), 2), '| all pairs:', round(forecast_2025.sum(), 2))
print('2024 long-run forecast added up:', round(forecast_long.values.sum(), 2), '| training units / 60 x 12:', round(train.values.sum() / 60 * 12, 2))

Delivered lines: 117656 | expected 117656
units on Delivered lines: 1234939 | expected 1234939
Delivered orders with at least one line: 18033 | Delivered orders in the header: 18033 | expected 18033 and 18033
units in the monthly table: 1234939
units in training plus test: 1234939
units by year added up: 1234939
products: 30 | largest difference by product: 0
branches: 6 | largest difference by branch: 0
pairs: 180 | largest difference by pair: 0
2025 forecast added up by product: 205823.17 | by branch: 205823.17 | all pairs: 205823.17
2024 long-run forecast added up: 207016.2 | training units / 60 x 12: 207016.2


In [19]:
# second method: add up units with a plain loop, without groupby or unstack
totals = {}
for pid, bid, d, q in zip(sales['product_id'], sales['branch_id'], sales['order_date'], sales['quantity']):
    key = (pid, bid, d.year, d.month)
    totals[key] = totals.get(key, 0) + q

differences = 0
for (pid, bid), row in monthly.iterrows():
    for period, value in row.items():
        if totals.get((pid, bid, period.year, period.month), 0) != value:
            differences += 1
print('cells compared:', monthly.shape[0] * monthly.shape[1], '| cells that differ:', differences, '| expected 0')
print('units in the loop totals:', sum(totals.values()), '| expected 1234939')
print('pair-months with sales in the loop:', len(totals), '| in the table:', int((monthly > 0).sum().sum()), '| expected equal')

# long-run forecast scores for 2024, worked out with plain loops
train_sum = {}
test_months = {}
for (pid, bid, y, m), v in totals.items():
    if y <= 2023:
        train_sum[(pid, bid)] = train_sum.get((pid, bid), 0) + v
    else:
        test_months.setdefault((pid, bid), [0] * 12)[m - 1] += v

gap_month = 0.0
gap_quarter = 0.0
gap_year = 0.0
actual_total = 0
product_forecast = {}
product_actual = {}
for pair, s in train_sum.items():
    f = s / 60
    actual = test_months.get(pair, [0] * 12)
    actual_total += sum(actual)
    gap_month += sum(abs(a - f) for a in actual)
    for i in range(0, 12, 3):
        gap_quarter += abs(sum(actual[i:i + 3]) - 3 * f)
    gap_year += abs(sum(actual) - 12 * f)
    product_forecast[pair[0]] = product_forecast.get(pair[0], 0) + 12 * f
    product_actual[pair[0]] = product_actual.get(pair[0], 0) + sum(actual)

gap_product = sum(abs(product_forecast[p] - product_actual[p]) for p in product_forecast)
print('actual 2024 units:', actual_total, '| expected 199858')
print('gap by month, percent of actual:', round(gap_month / actual_total * 100, 2), '| expected 32.2')
print('gap by quarter, percent of actual:', round(gap_quarter / actual_total * 100, 2), '| expected 18.7')
print('gap for the full year per pair, percent of actual:', round(gap_year / actual_total * 100, 2), '| expected 11.06')
print('gap for the full year per product, percent of actual:', round(gap_product / actual_total * 100, 2), '| expected 5.18')
print('2025 forecast total:', round(sum(totals.values()) / 72 * 12, 2), '| expected 205823.17')

cells compared: 12960 | cells that differ: 0 | expected 0
units in the loop totals: 1234939 | expected 1234939
pair-months with sales in the loop: 12955 | in the table: 12955 | expected equal
actual 2024 units: 199858 | expected 199858
gap by month, percent of actual: 32.2 | expected 32.2
gap by quarter, percent of actual: 18.7 | expected 18.7
gap for the full year per pair, percent of actual: 11.06 | expected 11.06
gap for the full year per product, percent of actual: 5.18 | expected 5.18
2025 forecast total: 205823.17 | expected 205823.17


In [20]:
# hide the last 6, 12 and 24 months in turn and score the three simple methods
def window_scores(window):
    cut = 72 - window
    train_part = monthly.iloc[:, :cut]
    test_part = monthly.iloc[:, cut:].values
    forecasts = {
        'recent level': np.repeat(train_part.iloc[:, -12:].mean(axis=1).values[:, None], window, axis=1),
        'same as last year': monthly.iloc[:, [cut - 12 + (j % 12) for j in range(window)]].values.astype(float),
        'long-run level': np.repeat(train_part.mean(axis=1).values[:, None], window, axis=1),
    }
    total = test_part.sum()
    scores = {}
    for name, f in forecasts.items():
        by_month = np.abs(test_part - f).sum() / total * 100
        by_window = np.abs(test_part.sum(axis=1) - f.sum(axis=1)).sum() / total * 100
        scores[name] = (round(by_month, 2), round(by_window, 2))
    return total, scores

for window in [6, 12, 24]:
    total, scores = window_scores(window)
    print('hidden months:', window, '| hidden units:', total)
    for name, (by_month, by_window) in scores.items():
        print('  ', name, '| gap by month:', by_month, '| gap for the whole window per pair:', by_window)
print('earlier 12-month results: recent 33.6 and 14.41, same as last year 44.42 and 14.41, long-run 32.2 and 11.06')

# the five biggest pair-months against the pair's own average month
stacked = monthly.stack()
pair_average = monthly.mean(axis=1)
for (pid, bid, month), value in stacked.sort_values(ascending=False).head(5).items():
    print(pid, bid, month, '| units:', value, '| pair average month:', round(pair_average.loc[(pid, bid)], 1), '| times the average:', round(value / pair_average.loc[(pid, bid)], 2))

hidden months: 6 | hidden units: 100993
   recent level | gap by month: 32.61 | gap for the whole window per pair: 15.6
   same as last year | gap by month: 43.96 | gap for the whole window per pair: 19.72
   long-run level | gap by month: 31.67 | gap for the whole window per pair: 13.08
hidden months: 12 | hidden units: 199858
   recent level | gap by month: 33.6 | gap for the whole window per pair: 14.41
   same as last year | gap by month: 44.42 | gap for the whole window per pair: 14.41
   long-run level | gap by month: 32.2 | gap for the whole window per pair: 11.06
hidden months: 24 | hidden units: 404496
   recent level | gap by month: 32.96 | gap for the whole window per pair: 10.81
   same as last year | gap by month: 45.03 | gap for the whole window per pair: 10.81
   long-run level | gap by month: 32.18 | gap for the whole window per pair: 7.75
earlier 12-month results: recent 33.6 and 14.41, same as last year 44.42 and 14.41, long-run 32.2 and 11.06
P001 CHN001 2019-07 | un

In [21]:
import numpy as np

actual = test.values
actual_total = actual.sum()

def gaps(forecast):
    f = np.asarray(forecast, dtype=float)
    month_gap = np.abs(f - actual).sum() / actual_total * 100
    f_q = f.reshape(180, 4, 3).sum(axis=2)
    a_q = actual.reshape(180, 4, 3).sum(axis=2)
    quarter_gap = np.abs(f_q - a_q).sum() / actual_total * 100
    year_gap = np.abs(f.sum(axis=1) - actual.sum(axis=1)).sum() / actual_total * 100
    return f.sum(), f.sum() - actual_total, month_gap, quarter_gap, year_gap

average_level = np.repeat(train.mean(axis=1).values[:, None], 12, axis=1)
median_level = np.repeat(train.median(axis=1).values[:, None], 12, axis=1)

print("level used | forecast total | minus actual | gap by month | gap by quarter | gap full year per pair")
for name, fc in [("average of 60 months", average_level), ("median of 60 months", median_level)]:
    total, diff, m, q, y = gaps(fc)
    print(name, "|", round(total), "|", round(diff), "|", round(m, 2), "|", round(q, 2), "|", round(y, 2))

level used | forecast total | minus actual | gap by month | gap by quarter | gap full year per pair
average of 60 months | 207016 | 7158 | 32.2 | 18.7 | 11.06
median of 60 months | 199830 | -28 | 32.01 | 18.61 | 11.17


In [22]:
months = monthly.columns
expected_months = pd.period_range("2019-01", "2024-12", freq="M")

print("months in the monthly table:", len(months))
print("first month:", months[0], "| last month:", months[-1])
print("months match the full list 2019-01 to 2024-12 with none missing:", list(months) == list(expected_months))

print("first order date:", sales["order_date"].min().date(), "| last order date:", sales["order_date"].max().date())

month_totals = monthly.sum(axis=0)
print("months with zero total units:", int((month_totals == 0).sum()))
print("smallest month total:", int(month_totals.min()), "in", month_totals.idxmin())
print("hidden test months (2024) with zero total units:", int((month_totals.iloc[-12:] == 0).sum()))

order_days = sales.groupby(sales["order_date"].dt.to_period("M"))["order_date"].apply(lambda s: s.dt.day.nunique())
print("fewest distinct order days in any month:", int(order_days.min()), "in", order_days.idxmin())
print("most distinct order days in any month:", int(order_days.max()), "in", order_days.idxmax())

print("first month of the 2025 forecast:", months[-1] + 1)
print("2025 forecast starts after the last order date:", (months[-1] + 1).start_time > sales["order_date"].max())

months in the monthly table: 72
first month: 2019-01 | last month: 2024-12
months match the full list 2019-01 to 2024-12 with none missing: True
first order date: 2019-01-01 | last order date: 2024-12-31
months with zero total units: 0
smallest month total: 14117 in 2019-02
hidden test months (2024) with zero total units: 0
fewest distinct order days in any month: 28 in 2019-02
most distinct order days in any month: 31 in 2019-01
first month of the 2025 forecast: 2025-01
2025 forecast starts after the last order date: True


In [23]:
order_info = {}
for so_id, order_date, branch in zip(delivered["so_id"], pd.to_datetime(delivered["order_date"]), delivered["branch_id"]):
    order_info[so_id] = (order_date, branch)

raw_units = {}
for so_id, prod, qty in zip(so_l["so_id"], so_l["product_id"], so_l["quantity"]):
    if so_id in order_info:
        order_date, branch = order_info[so_id]
        key = (prod, branch, order_date.year, order_date.month)
        raw_units[key] = raw_units.get(key, 0) + qty

pairs_to_check = [("P004", "HYD001"), ("P019", "AHM001"), ("P001", "CHN001")]

for prod, branch in pairs_to_check:
    train_units = sum(raw_units.get((prod, branch, y, m), 0) for y in range(2019, 2024) for m in range(1, 13))
    actual_2024 = sum(raw_units.get((prod, branch, 2024, m), 0) for m in range(1, 13))
    average_month = train_units / 60
    forecast_year = average_month * 12
    gap_units = abs(forecast_year - actual_2024)
    quiet_months = [(y, m) for y in range(2019, 2025) for m in range(1, 13) if raw_units.get((prod, branch, y, m), 0) == 0]

    table_average = train.loc[(prod, branch)].mean()
    table_actual = test.loc[(prod, branch)].sum()
    table_forecast = forecast_long.loc[(prod, branch)].sum()

    print("pair:", prod, branch)
    print("  from raw files: units in 60 training months", int(train_units), "| average month", round(average_month, 2))
    print("  from raw files: 2024 actual", int(actual_2024), "| forecast for 2024", round(forecast_year, 2), "| gap", round(gap_units, 2), "units =", round(gap_units / actual_2024 * 100, 2), "% of this pair's 2024 actual")
    print("  months with zero units:", quiet_months)
    print("  tables: average month", round(table_average, 2), "| 2024 actual", int(table_actual), "| forecast for 2024", round(table_forecast, 2))
    print("  raw files and tables match:", abs(average_month - table_average) < 1e-6 and actual_2024 == table_actual and abs(forecast_year - table_forecast) < 1e-6)

pair: P004 HYD001
  from raw files: units in 60 training months 4383 | average month 73.05
  from raw files: 2024 actual 765 | forecast for 2024 876.6 | gap 111.6 units = 14.59 % of this pair's 2024 actual
  months with zero units: [(2021, 1), (2024, 6)]
  tables: average month 73.05 | 2024 actual 765 | forecast for 2024 876.6
  raw files and tables match: True
pair: P019 AHM001
  from raw files: units in 60 training months 6252 | average month 104.2
  from raw files: 2024 actual 928 | forecast for 2024 1250.4 | gap 322.4 units = 34.74 % of this pair's 2024 actual
  months with zero units: []
  tables: average month 104.2 | 2024 actual 928 | forecast for 2024 1250.4
  raw files and tables match: True
pair: P001 CHN001
  from raw files: units in 60 training months 7429 | average month 123.82
  from raw files: 2024 actual 1252 | forecast for 2024 1485.8 | gap 233.8 units = 18.67 % of this pair's 2024 actual
  months with zero units: []
  tables: average month 123.82 | 2024 actual 1252 | 